Run this lab notebook in VS Code or upload the file to Google Colab. It uses CPU computation. The first setup cell installs any missing Python libraries. All required sequence data are included in the file. The optional HMMER activity uses its website.

[Session 4 chapter](https://course.bioinfo.se/session04.html)


# Session 4 sequence alignment lab

Complete four stages and use your results in the ten-question lab quiz.

1. Fill the global-alignment matrix by hand, trace back an optimal alignment, and calculate identity and similarity.
2. Compare scoring choices and interpret a supplied shuffle example.
3. Build a progressive multiple alignment of three globin windows.
4. Calculate column entropy and read a logo from the included Pfam PF00042.29 snapshot.


The Canvas module gives current deadlines and assessment arrangements. Work through each stage before running the later cells that calculate its results.


In [ ]:
import importlib.util
import subprocess
import sys

required = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("Bio", "biopython"),
    ("logomaker", "logomaker"),
]
missing = [package for module, package in required
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


## Setup: the exercise for Step 1

Align two real, short protein fragments by hand:

- **seqA = "HFGKE"** -- human beta-globin (HBB, UniProt `P68871`), residues 118-122
- **seqB = "HGQE"** -- human myoglobin (MYG, UniProt `P02144`), residues 25-28

Use a flat, linear gap penalty of **-4** per gap position, and the real
**BLOSUM62** substitution matrix (Henikoff & Henikoff, 1992) below,
restricted to just the residues that appear in these two fragments:

|     | E  | F  | G  | H  | K  | Q  |
|-----|---:|---:|---:|---:|---:|---:|
| **E** | 5 | -3 | -2 | 0 | 1 | 2 |
| **F** | -3 | 6 | -3 | -1 | -3 | -3 |
| **G** | -2 | -3 | 6 | -2 | -2 | -2 |
| **H** | 0 | -1 | -2 | 8 | -1 | 0 |
| **K** | 1 | -3 | -2 | -1 | 5 | 1 |
| **Q** | 2 | -3 | -2 | 0 | 1 | 5 |

The recurrence (same as the book page and the teaching notebook):

$$
S[i,j] = \max \begin{cases} S[i-1,j-1] + s(x_i, y_j) & \text{diagonal: match/mismatch} \\ S[i-1,j] + \text{gap} & \text{up: gap in seqB} \\ S[i,j-1] + \text{gap} & \text{left: gap in seqA} \end{cases}
$$

Run the setup cell below to get these same values programmatically (and
the mechanical border row/column already filled in) -- reading them
here first just means you don't have to execute anything to know what
the exercise even is.

The interior has 5 × 4 cells. Including the empty-prefix border, the matrix has 6 × 5 cells. Global traceback must reach (0,0). The residue positions above use UniProt sequence numbering, including the initial methionine where present.


In [ ]:
import numpy as np

# Two real protein fragments -- not the same ones used anywhere else on
# the Session 4 page or its teaching notebook:
#   seqA = "HFGKE", human beta-globin (HBB, UniProt P68871), residues 118-122
#   seqB = "HGQE",  human myoglobin  (MYG, UniProt P02144), residues 25-28
seqA, seqB = "HFGKE", "HGQE"
gap = -4.0

# BLOSUM62 restricted to the residues in this hand exercise.
BLOSUM62_SUB = {
    ('E', 'E'): 5, ('E', 'F'): -3, ('E', 'G'): -2, ('E', 'H'): 0, ('E', 'K'): 1, ('E', 'Q'): 2,
    ('F', 'E'): -3, ('F', 'F'): 6, ('F', 'G'): -3, ('F', 'H'): -1, ('F', 'K'): -3, ('F', 'Q'): -3,
    ('G', 'E'): -2, ('G', 'F'): -3, ('G', 'G'): 6, ('G', 'H'): -2, ('G', 'K'): -2, ('G', 'Q'): -2,
    ('H', 'E'): 0, ('H', 'F'): -1, ('H', 'G'): -2, ('H', 'H'): 8, ('H', 'K'): -1, ('H', 'Q'): 0,
    ('K', 'E'): 1, ('K', 'F'): -3, ('K', 'G'): -2, ('K', 'H'): -1, ('K', 'K'): 5, ('K', 'Q'): 1,
    ('Q', 'E'): 2, ('Q', 'F'): -3, ('Q', 'G'): -2, ('Q', 'H'): 0, ('Q', 'K'): 1, ('Q', 'Q'): 5,
}
blosum62 = BLOSUM62_SUB
residues = sorted(set(seqA) | set(seqB))

def score(a, b, subst=blosum62, gap_penalty=gap):
    if a == '-' or b == '-':
        return gap_penalty
    return subst[a, b]

print(f"seqA = {seqA!r}  ({len(seqA)} residues, human beta-globin fragment)")
print(f"seqB = {seqB!r}  ({len(seqB)} residues, human myoglobin fragment)")
print(f"gap penalty = {gap} (flat, linear -- same recurrence as the book page and teaching notebook)")
print()
print("BLOSUM62, restricted to just the residues appearing above (real values):")
print("     " + "  ".join(f"{r:>2}" for r in residues))
for r1 in residues:
    print(f"{r1:>2}  " + "  ".join(f"{blosum62[r1, r2]:>2}" for r2 in residues))
print()

m, n = len(seqA) + 1, len(seqB) + 1
S = np.full((m, n), np.nan)
for i in range(m):
    S[i, 0] = i * gap
for j in range(n):
    S[0, j] = j * gap

print("Border filled in (mechanical -- i*gap / j*gap), interior left for you to work out by hand:")
print("Rows = '-' + seqA, columns = '-' + seqB:")
print("        " + "     ".join(["-"] + list(seqB)))
for i, row_label in enumerate(["-"] + list(seqA)):
    print(row_label, " ", S[i])


## Step 1: fill in the matrix, one row at a time

Work out each row of the interior matrix by hand, then fill in that
row's cell below and run it to check just that row -- rather than
committing to all 20 values before getting any feedback at all. Row $i$
corresponds to `seqA[i-1]`, e.g. row 1 is `seqA[0] = 'H'`.


In [ ]:
# Fill the four interior values in row 1.
row_1_guesses = {
    (1, 1): None, (1, 2): None, (1, 3): None, (1, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            real[i, j] = max(
                real[i - 1, j - 1] + score(seqA[i - 1], seqB[j - 1]),
                real[i - 1, j] + gap,
                real[i, j - 1] + gap,
            )
    return real

def check_row(guesses):
    expected = _real_matrix()
    correct = True
    for (i, j), guess in guesses.items():
        if guess is None:
            print(f"S[{i},{j}]: enter your value")
            correct = False
        elif np.isclose(float(guess), expected[i, j]):
            print(f"S[{i},{j}]: correct")
        else:
            print(f"S[{i},{j}]: check the three candidates")
            correct = False
    return correct

row_1_correct = check_row(row_1_guesses)


In [ ]:
# Fill the four interior values in row 2.
row_2_guesses = {
    (2, 1): None, (2, 2): None, (2, 3): None, (2, 4): None,
}

row_2_correct = check_row(row_2_guesses)


In [ ]:
# Fill the four interior values in row 3.
row_3_guesses = {
    (3, 1): None, (3, 2): None, (3, 3): None, (3, 4): None,
}

row_3_correct = check_row(row_3_guesses)


In [ ]:
# Fill the four interior values in row 4.
row_4_guesses = {
    (4, 1): None, (4, 2): None, (4, 3): None, (4, 4): None,
}

row_4_correct = check_row(row_4_guesses)


In [ ]:
# Fill the four interior values in row 5.
row_5_guesses = {
    (5, 1): None, (5, 2): None, (5, 3): None, (5, 4): None,
}

row_5_correct = check_row(row_5_guesses)


### Trace back the alignment

Trace back from the bottom-right cell to (0,0), including any moves along the top row or left column. Write out the two aligned strings, then use the next cell to check your result.


In [ ]:
# Enter your two aligned strings, using "-" for each gap.
your_alignment = (None, None)

def check_alignment(candidate):
    if len(candidate) != 2 or not all(isinstance(row, str) for row in candidate):
        print("Enter both aligned strings.")
        return False
    aligned_a, aligned_b = candidate
    valid = (
        len(aligned_a) == len(aligned_b)
        and aligned_a.replace("-", "") == seqA
        and aligned_b.replace("-", "") == seqB
        and all(a != "-" or b != "-" for a, b in zip(aligned_a, aligned_b))
    )
    if not valid:
        print("Check the sequence order, equal row lengths and gap placement.")
        return False
    candidate_score = sum(score(a, b) for a, b in zip(aligned_a, aligned_b))
    optimal = _real_matrix()[-1, -1]
    correct = bool(np.isclose(candidate_score, optimal))
    print("Optimal under these settings." if correct else "Revisit the traceback choices.")
    return correct

alignment_correct = check_alignment(your_alignment)


When the row checks and alignment check succeed, record the final matrix cell and your aligned strings. A tied optimum can have several valid tracebacks. The checker accepts any valid alignment with the optimal score.


### Identity and similarity by hand

Use the alignment you just traced back.

1. Count the columns containing two identical residues.
2. Use the BLOSUM62 table to count the columns containing two residues with a **strictly positive** substitution score. This is the rule for similarity in this lab. An identical pair also counts here if its score is positive.
3. Divide each count by the **total number of alignment columns**, including gaps, and multiply by 100. A gap column contributes only to the denominator.

Record both counts, the total and the percentages. Keep your values for the comparison in Step 2.


## Step 2 scoring choices

Run the same fragments with different linear gap scores and substitution matrices. Before running the cell, predict what will happen to the alignment, raw score, percent identity and percent similarity when the linear gap score changes from −4 to −8. Compare the −4 row with your hand calculation. The table applies the same denominator and positive BLOSUM62 rule at each gap setting.

The second comparison varies the substitution matrix. Raw scores from different scoring schemes have different scales.

For an affine gap, use g(n) = g_open + (n−1) g_extend for each contiguous run. Score VMMMVAH--FG against VM--VIHWHYG by hand, using match +2, mismatch −1, gap opening −4 and extension −1. Record each gap run separately for the lab quiz.


In [ ]:
def global_align(seqA, seqB, gap, subst):
    """Same Needleman-Wunsch recurrence as Step 1 and the teaching notebook,
    generalized to take any gap penalty / substitution matrix."""
    m, n = len(seqA) + 1, len(seqB) + 1
    S = np.zeros((m, n))
    trace = np.zeros((m, n, 2), dtype=int)
    for i in range(1, m):
        S[i, 0] = i * gap
        trace[i, 0, :] = (-1, 0)
    for j in range(1, n):
        S[0, j] = j * gap
        trace[0, j, :] = (0, -1)
    for i in range(1, m):
        for j in range(1, n):
            diag = S[i-1, j-1] + subst(seqA[i-1], seqB[j-1])
            up = S[i-1, j] + gap
            left = S[i, j-1] + gap
            S[i, j] = max(diag, up, left)
            if diag >= max(up, left):
                trace[i, j, :] = (-1, -1)
            elif up >= left:
                trace[i, j, :] = (-1, 0)
            else:
                trace[i, j, :] = (0, -1)
    return S, trace

def traceback(seqA, seqB, S, trace):
    i, j = len(seqA), len(seqB)
    outA, outB = "", ""
    while i > 0 or j > 0:
        di, dj = trace[i, j]
        i, j = i + int(di), j + int(dj)
        outA = ("-" if di == 0 else seqA[i]) + outA
        outB = ("-" if dj == 0 else seqB[j]) + outB
    return outA, outB

# Load the standard tables rather than maintaining duplicate wide tables.
from Bio.Align import substitution_matrices
PAM250_SUB = substitution_matrices.load("PAM250")
BLOSUM62_WIDE = substitution_matrices.load("BLOSUM62")

def identity_score(a, b):
    return 1.0 if a == b else -1.0

def blosum62_score(a, b):
    return BLOSUM62_WIDE[a, b]

def pam250_score(a, b):
    return PAM250_SUB[a, b]

print("Varying the gap score with BLOSUM62 fixed")
print(f"{'Gap score':>9} {'Score':>6} {'Identity':>14} {'Similarity':>14}   Alignment")
for gap_test in [-1.0, -4.0, -8.0, -12.0]:
    scores, moves = global_align(seqA, seqB, gap_test, blosum62_score)
    aligned_a, aligned_b = traceback(seqA, seqB, scores, moves)
    columns = len(aligned_a)
    residue_pairs = [(a, b) for a, b in zip(aligned_a, aligned_b) if a != "-" and b != "-"]
    identical = sum(a == b for a, b in residue_pairs)
    positive = sum(blosum62_score(a, b) > 0 for a, b in residue_pairs)
    identity = f"{identical}/{columns} ({identical / columns:.1%})" if columns else "undefined"
    similarity = f"{positive}/{columns} ({positive / columns:.1%})" if columns else "undefined"
    print(f"{gap_test:>9.0f} {scores[-1, -1]:>6.0f} {identity:>14} {similarity:>14}   "
          f"{aligned_a} / {aligned_b}")

print()
print("Varying the substitution matrix (gap=-4 fixed):")
for label, subst in [("identity", identity_score), ("BLOSUM62", blosum62_score), ("PAM250", pam250_score)]:
    S_t, tr_t = global_align(seqA, seqB, -4.0, subst)
    oA, oB = traceback(seqA, seqB, S_t, tr_t)
    print(f"  {label:>8s}  score={S_t[-1,-1]:>5.0f}   {oA} / {oB}")


Compare the −4 and −8 rows. Identify which quantities changed. Use the aligned strings and the number of gap positions to explain the raw score difference, then answer Q5.

### Interpreting a background comparison

Suppose a protein alignment scores 48. In 199 shuffles of one sequence, scored with the same method and settings, 40 scores reach 48 or higher. Use this supplied example to explain how unusual the observation is under that shuffle procedure.

The count describes this particular background experiment. A database-search E-value also needs the search space and the search program’s statistical model. Record the sequence, scoring scheme and shuffle procedure when reporting an empirical comparison.

### Optional live HMMER search

Open [HMMER at EMBL-EBI](https://www.ebi.ac.uk/Tools/hmmer/) and use phmmer to compare the HBB sequence below with Swiss-Prot. Record the date, database or release information, query, settings and returned output. Inspect the current support for alternative scoring settings in the service. Explain what an E-value measures and how the size of the search space affects it.

The required quiz uses the fixed inputs and supplied shuffle example. Live hit counts and ranks can vary with database releases and service settings.


In [ ]:
HBB = (
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTF"
    "ATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"
)
print(">HBB_human_P68871")
print(HBB)


## Step 3 progressive multiple alignment

The three 20-residue windows below come from HBA1, MB and HBB. Use BLOSUM62 with a linear gap score of −6. Select the first pair by the largest pairwise score, then align the third sequence to the resulting profile.

A profile column contributes its mean pair score against the added residue. A residue-gap pair scores −6 and a gap-gap pair scores zero. If the merge creates a new column, it adds a gap to every existing profile row. This preserves the column correspondence already chosen.

The first merge concerns these windows. Explain why its order alone cannot establish the evolutionary history of the full genes.


In [ ]:
def add_sequence_to_profile(profile, new_name, sequence, gap, substitution):
    """Align a sequence to fixed profile columns with mean pair scores.

    A residue-gap pair scores gap. A gap-gap pair scores zero.
    Existing profile columns stay together. New columns add gaps to
    every existing row. This is a progressive heuristic.
    """
    rows = list(profile.values())
    if not rows or len({len(row) for row in rows}) != 1:
        raise ValueError("The profile must contain equally long aligned rows.")
    if new_name in profile:
        raise ValueError("The new sequence needs a distinct name.")
    width, length = len(rows[0]), len(sequence)
    columns = list(zip(*rows))

    def column_pair(column, residue):
        return sum(gap if old == "-" else substitution(old, residue)
                   for old in column) / len(rows)

    def column_gap(column):
        return gap * sum(old != "-" for old in column) / len(rows)

    scores = np.zeros((width + 1, length + 1))
    moves = np.zeros((width + 1, length + 1, 2), dtype=int)
    for i in range(1, width + 1):
        scores[i, 0] = scores[i - 1, 0] + column_gap(columns[i - 1])
        moves[i, 0] = (-1, 0)
    for j in range(1, length + 1):
        scores[0, j] = j * gap
        moves[0, j] = (0, -1)
    for i in range(1, width + 1):
        for j in range(1, length + 1):
            candidates = [
                scores[i - 1, j - 1] + column_pair(columns[i - 1], sequence[j - 1]),
                scores[i - 1, j] + column_gap(columns[i - 1]),
                scores[i, j - 1] + gap,
            ]
            choice = int(np.argmax(candidates))
            scores[i, j] = candidates[choice]
            moves[i, j] = [(-1, -1), (-1, 0), (0, -1)][choice]

    result = {name: [] for name in profile}
    result[new_name] = []
    i, j = width, length
    while i > 0 or j > 0:
        di, dj = moves[i, j]
        for name, row in profile.items():
            result[name].append(row[i - 1] if di else "-")
        result[new_name].append(sequence[j - 1] if dj else "-")
        i, j = i + int(di), j + int(dj)
    return {name: "".join(reversed(row)) for name, row in result.items()}

MSA_WINDOWS = {
    "HBA1": "TNVKAAWGKVGAHAGEYGAE",  # P69905 residues 9–28
    "MB": "QLVLNVWGKVEADIPGHGQE",    # P02144 residues 9–28
    "HBB": "SAVTALWGKVNVDEVGGEAL",   # P68871 residues 10–29
}
names = list(MSA_WINDOWS)
pair_scores = {}
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        scores_pair, trace_pair = global_align(MSA_WINDOWS[a], MSA_WINDOWS[b], -6.0, blosum62_score)
        pair_scores[a, b] = float(scores_pair[-1, -1])
        print(f"{a}–{b}: {scores_pair[-1, -1]:.0f}")

best_pair = max(pair_scores, key=pair_scores.get)
n1, n2 = best_pair
n3 = next(name for name in names if name not in best_pair)
a, b = n1, n2
scores_pair, trace_pair = global_align(MSA_WINDOWS[a], MSA_WINDOWS[b], -6.0, blosum62_score)
aligned_a, aligned_b = traceback(MSA_WINDOWS[a], MSA_WINDOWS[b], scores_pair, trace_pair)
profile = {n1: aligned_a, n2: aligned_b}
msa_result = add_sequence_to_profile(
    profile, n3, MSA_WINDOWS[n3], -6.0, blosum62_score
)
print(f"First merge: {n1} and {n2}")
for name, row in msa_result.items():
    print(f"{name:5s} {row}")


### Gap propagation in a small example

The globin windows above align without new gap columns under these settings. To see how a profile merge handles an insertion, use the short teaching strings below. Compare the two old rows before and after adding the third sequence. A newly inserted column adds a gap to both old rows.


In [ ]:
toy_profile = {"first": "A-C", "second": "AGC"}
toy_result = add_sequence_to_profile(
    toy_profile, "third", "ATGC", -2.0,
    lambda a, b: 2.0 if a == b else -1.0,
)
print("Before adding the third sequence:")
for name, row in toy_profile.items():
    print(name, row)
print("After merging:")
for name, row in toy_result.items():
    print(name, row)
inserted_columns = [
    i + 1 for i in range(len(toy_result["first"]))
    if all(toy_result[name][i] == "-" for name in toy_profile)
]
print("New columns with gaps in both old rows:", inserted_columns)


In [ ]:
import matplotlib.pyplot as plt

names_order = list(msa_result)
ncols = len(next(iter(msa_result.values())))

fig, ax = plt.subplots(figsize=(0.42 * ncols + 1.5, 0.6 * len(names_order) + 1))
for row, name in enumerate(names_order):
    seq = msa_result[name]
    for col, ch in enumerate(seq):
        others = [msa_result[n][col] for n in names_order if n != name]
        if ch == '-':
            color = "#f0f0f0"
        elif all(ch == o for o in others):
            color = "#8fd18f"
        else:
            color = "#ffe08a"
        ax.add_patch(plt.Rectangle((col, len(names_order) - 1 - row), 1, 1,
                                    facecolor=color, edgecolor="white"))
        ax.text(col + 0.5, len(names_order) - 1 - row + 0.5, ch,
                ha="center", va="center", fontsize=11, family="monospace")
ax.set_xlim(0, ncols)
ax.set_ylim(0, len(names_order))
ax.set_yticks([len(names_order) - 1 - i + 0.5 for i in range(len(names_order))])
ax.set_yticklabels(names_order)
ax.set_xticks([])
ax.set_title("Progressive MSA: real 20-residue windows around the conserved WGKV motif\n"
              "(green = fully conserved column, yellow = mismatch, grey = gap)")
plt.tight_layout()
plt.show()


## Step 4 entropy and sequence logos

Entropy H = −sum(p log2 p) describes residue variation within a column. Work with the non-gap residues. Under an equal protein background, the simple information content is log2(20) − H.

The logo below multiplies this by occupancy, the fraction of sequences with a residue in that column. Each letter’s height is its frequency multiplied by the stack height. This teaching plot applies occupancy scaling and omits sequence weighting and small-sample corrections.

### Entropy by hand

The next cell selects columns with all residues identical, two matching residues and one different residue, and three different residues. Calculate their entropy before entering your guesses.


In [ ]:
# Pick three columns from the Step 3 MSA (msa_result), by how many different residues they contain
columns = ["".join(msa_result[n][c] for n in msa_result) for c in range(len(next(iter(msa_result.values()))))]
picked = {}
for c, col in enumerate(columns):
    if "-" in col:
        continue
    kind = {1: "all same", 2: "two same, one different", 3: "all different"}[len(set(col))]
    picked.setdefault(kind, (c + 1, col))
for kind in ["all same", "two same, one different", "all different"]:
    c, col = picked[kind]
    print(f"column {c:2d}: {' '.join(col)}   ({kind})")

In [ ]:
entropy_guesses = {
    "all same": None,
    "two same, one different": None,
    "all different": None,
}

def column_entropy(column):
    residues = [residue for residue in column if residue != "-"]
    if not residues:
        return 0.0
    frequencies = np.array([residues.count(r) for r in set(residues)]) / len(residues)
    return float(-(frequencies * np.log2(frequencies)).sum()) + 0.0

entropy_correct = {}
for kind, guess in entropy_guesses.items():
    expected = column_entropy(picked[kind][1])
    correct = guess is not None and abs(float(guess) - expected) <= 0.01
    entropy_correct[kind] = correct
    print(f"{kind}: " + ("enter your value" if guess is None else
                        "correct" if correct else "check your frequencies and logarithms"))


Three equally weighted, non-gap residues can show at most log2(3) bits of observed entropy. A protein alphabet with 20 equally frequent residues has maximum entropy log2(20).

### The saved globin family alignment

This notebook includes the original Stockholm snapshot PF00042.29, with 73 sequences and 141 alignment columns. The source is the Pfam Globin seed alignment distributed through InterPro. Its version and checksum identify the dataset used for the quiz. The next cell reads the included data through Biopython’s Stockholm parser.


In [ ]:
import base64
import gzip
import hashlib
import io
from Bio import AlignIO

# Pfam Globin PF00042.29 seed alignment, from the course's saved InterPro snapshot.
# Provider terms: https://www.ebi.ac.uk/about/terms-of-use
SEED_SHA256 = "31ae1ef89b410e0a8e5c2aaa765ed0af5da3cb3499cba7823f121d749b12504e"
SEED_GZIP_BASE64 = "H4sIAAAAAAAC/818XXPbuLblu3+Fqu7b3A4jybZsn9Q8wMQH0QAREkCgw0xNudyOu+26ju0T2XOm76+ftUFSohz59MzDVF12xx2TELiwsfbaewNQ/9ssxM+lqT7berYo5kf/9t+VnGk+m83Uw9Nv94/9DVbiRiPn8/nJslhe9De5eNOq0bjxR77xafjcF9y5vH65/X79OGOf8Pn5B/xZflhdnC8/nKxW80m78u7p5e7+elYOHw7Uf3j58Xrz8vrj9mrz+nz74/lpc/9y/zS8UDG0WC6L+bz/OXwwlgdvu3x7UVzM+5/D7csat+++f//t9f7h22xW1XVx/fiItwtOfxmwDI02t9c/bu5mH77OLuZnJ6vT5WL24cPN8+vsfPZBzBYYG3Uwe/79Gk3/McBp8Fn+9P16tNPavDFcaWn4dj6/GOzhHW78j8X/HH6jlx+vTlcnJyfDnUhd3r7c/vh+/3j9+LKZPf0+u549/3h6ub1/nP3+9PCtmH15vP/H6+3s99trsl9u8nJ3O0zQpJ9r9PE0u765/zYD5tfbx5vbTTE8Z3n+Nne/P/34NuO/7Obol5m93fzHjNWDFT0N4ddZ/fQwu7zHj8XF+dmnxcXqb4uLiw/LxWrskOagfHp8gTE2s/tvt48v97/f31zTnPZjeHz6X7cPM9jv/uFPuvMI7rw8fRtxb6YGWk4NdLGYny9OVpOBif/98uP65uX+8Y+tZa4f7v94/I63zr6jz4fN7PcfT9+zWcahz75dv1z/dr25nZrA3b7+8xrsYBLjvn+d/Rrov8/Eav4r/nr9zx/5s6WYmsO93jzc3t/MGEy7mXlMAYxx9ml5+rfFanX6YbE6OyumwzmeDmd5cnK+Or6YDEe+Pt6QnWg8d9e33596i8z+uH1E1/jL49Pjh8enb68P1/2YH4gZezP59Mfr5j9pHtnz88Ptb3/OSkboH79tvhPnMRR++/h4v5kJDDD+uH+8v/mPWf3rL5M+mtvrmyfcXf+6N9TMMZr180/Hx4u/Lc7OPyzO53vDO5kOb7W8WJ6fzCfD++4d+/DtFmy4p/n57fblevb04eXu+uF6s7n9DtKBxK8PL8OcXc82GOTD7aSHR7L30wtYNcPc3pKtpqM39xjab68//ribMYy0vv7H6/XLzIpfZuHm7vbbwyz+MvPXN3dg3u0/71/+c4Yn/mmzmf36V5N6vvy0mP/t9GS5+IAf46Ryj+baReEb//nTTDce8nBxtvg0eYwnQUfxadbwz+V8fnZx/C+eLs7fPEXnAQ89PTofH5GH5akfPAZ2+uMxT887n10tFtPP3v748+Xux9PNj9enH6QlvSf+dSfLaSfXD89317O7HUn/+vPH089///NffTCUn5tPs8Xdb9efZpvfpzapdY0HJ6DefNrdhEXTyak+1yF6xj9lU037CZgwilYni7NPs+enhz+fb5+JWFffso6PEapF07Nj+iXMlL1cXtkvdRQfjxcfgGE2XAiezXy5XJwXy0nL2NkKLZdoebzX8uJitSoW25bHV5bVofl4PH/b5+L0ZHUxaXkVv1zGLx+XFx8WxxezvZbny/ly0vJkxLlCn6u9lsenZxd7LQecxz/hXC6X8z2c77WkPs8nLRdjS4zo+OytlbZvrzt1VXZNyT4u0eXxavam5fxktCe1ZNbW+uPybPrybcv5tE/FrBr6PN+30snxxdm0z0rE5vOhltTnamxZXbL6irVfyvhxCXsuL/Zbou34drQ8vmqsWLMMdH/si8X5xWrS8utVyZpKH2p5fHY+efvliPP0gJUwpl2fl1eh/cLKvuXZfsuzk/nFpOXy6u/CWZZHdLJ80+fx8eTtC0ySqg+1XB7PF2dTnFY0Dctv3zXc9nk+xTnM5umU8tuW80nL+VX9+UsQh3AiWTouTrct3RWra4uW86mZhjk6n++9PU/mgvzo9Ke3n05aLshKl5/fAB3sebycMGQc++oAl+bLydsZvOOzH+y5jxMNT04mLYd533e4bZ+Lacth3lc/MwTzPrE841fCd7U98PbFClq417Jhof58oCX6XBxP316yWA5dnr3Fudfn4kqrL1odevv5xdl07PGq+uxp3n/yjvnqbHEysfzi6vJzqL8cwon0dP/tw2z+3Cdmc77X53tePF8hZytOJlrXMF9WH4/R8nRfFU8Xq8XYZ3vGL/4OLndl/PxxBYaultuW/bMdzmYJgwZl88SfvvG4PYY0i6FllsX9sZ/PT7dWsgo4a8FDP6I3fnSxON9aKbe0XxoLe57/1OdyibEP0bCFkp+d4/1CWLjS4sPx4mQ7ovxs7JPN2dLOG/P3i6ukY1j0M3WcW+6ebUdVN1ei/Gx132xfHZYnsMDYskW6tFgMCE4RZ89OdzbNz6Yx6XiEim5hxEnLi9V8N0/Q/eNxVBcnKGzOdy3zs704p2zHdW+pn7KB1SQbaLZNLyZNhxldTLMBopMYkoGTvbGfz8+Wi0mfi/lVWelY9VN6+iZyT3Gejw1PD8X4s70R9S0zm0/etpy+/XhsSbr8NsYvL/YyoaElrHT6lk/L5aTl6l+2PJm0PNu+HvY8PZt63fL0ZDr2OI794uc4tzyeaKhcXH3xgpKRn2dztTiZ5nYLRBDHou/NtNyLScvTRbHLF69YY38dpOnip1zkeDrvpfD+Lcwtl06nLS+/BIJ5MTX8tuXJpOXyqql8+HwgawFflvtM3rV8o8snEJJJy2rwjZzX7s378Xy1PJtY6YqFMnz5eHK6E7uR8/DNqXdcNUHw8mDL1cXeiKqr6HX5uW+6N0fLs9UuJkFVjpuzsuRnV043l36rer3iDM8m/ZZDu5OTt/2eQtXJP99N/idXy/TasyhZmxoRfLDS+IKuVCnOY6gaJjWrmPDJFkrhgeWa6RAtbxsWrcEd2JZXtk1tJfCb8kY3nLpwneRMFiYybauUGGutKsbLlx33gq05KzUvKi240kfvFiGTSzO7NqEGYu4483yKWPBUBSAWrKqZT7JwPWKfdAg2ctEk63AHSERlfdtWGr8Zq3TMyKpaOlsXPirLumSbN4glN0DMWLnmIumu6REfLoYmVyX1OlZJGs6asIfYK+50FYHYw8ZxYuO6xDgt50L1iGNrWWVZaKuA34JXKbAij6TjKRWiCqmuVRKV0FvErUtAzNm6ZaXlBQKZ6hEfLsomlzV27YKWHoiF9mqHmAMx6wITZOM9VtRRB0LcMiZ5b2MgNkBMQIV32mTEvOsUZiY5YS2DjXk3sXE0lkvD1kGUuEuIWUZ8uDicXJ5hrGTZqmtiMMxKMSA2GtMughLShMomn9jICmu1I8TRJtuSVSurYGOeWdF6o1Ikeheik17pQgkmQSrbtMmGLeKilC7Bxg70qApvE/MD4kOl5xSxBUaBP5orFs3ExuiPR/BYCMmqZD3md0AcrQYfFA/4G7EiWA3PY6KtAJR5pqSJuYtOGiELBTs49NCIxPYRN+Cxq0DoQtnE08Djv0AMrZAtEDPLG8dgHql2iA0EkfG1AWINd9vamCXYWPAGUNpeKwwQ+7byRIqgUs8KIOaiLgSaWZNtHCaIK+mCglbAxrHwmg2IDxftU63gSTonpY8OkkTq5kbE4HHqYGOwraoTaQUjxJJrq0MCKwy0gnjcMvC4A6nJxkUrVdRNjwofhY3bWMs2aiYqXe0QK8Y1EHtWdpgenSI8793Fg8mlRLLRgwtt1YjYGisbIxPYxwIchinHGCxWgTHwPEOIrIIjGQYpyY5GnqetaSyLkW64WBlISdZjJ73VRcQrkkk1g7G3nmc5U6oNzHqfamhF0or3iA8uYkzVDVKoQeZWZMTCS2HghjVkHzNqQhCmxmiUi7BxZBmxA8LWKDciBiJCDAfFDUgzIkii+6azQuoiCK2BGDY23WB7xiVMHCA2tQdnRGGld7w+encxZcpjvN7CxqLtIAxt6GBja5yBO8GWhgM4tKplPNt4RJyMIcQDK9sG02AZi6QVkWzcZGRVJ6OBjUGIBFcNoh4R468OnbZQEXBZ4m26DIONDy7qTPWY9YgNTCOi4ZkVRuiiSS1ClHOCW28qCBYQ+z0bmxFxdMmAQTwjdmaL2DiZiBUOiOF5cYIYmq6auhVBBpPgeYkQ26N3F5cmFxOhFoTYEStMkDJ42XfLdWhUaCG/sVLKaPhQj1gwbYLlLgvFcJkODatIiBGUek3gIJe1RWBMawFcrJIDj+E9Lrss1M1IRBCREEAy4sOLXHueB10DYhYkfNrHToZqRGwJMVTHeEKc6N24rfERZirNQ80HVjAbgBKI8bxzkG6ebamcbAlxVeoWMY8B8Y7HsWlLQsyBmBUMgT1lxIcX2yZXFNGSjQNEpWUGiJsd4ioAcWuRVyhMXioYIU6KY5KDhvbFYY57xKJHnLaIk5OAQ4itTRGJxx5ixhH8e8Sh0AkqTojfWfSbsoJ8xtdSp65Zx+g7qVxOISEB6LUMSA3AcWYM8A2sSCxFBGYYf0DcyogJMYYQI5xUiF+5gyCJFcyUMiWBlM4OPReAyZo2UqoLAQFXUkLE1kfvLj6+QUw2Thkx2TiZbGOnjKxY4CGmFkKryMaZFTYm1jmwYosYswJAhLgbEWdbCtcjdqUVhLhidoggCN4RQZtYEYwMrkecbXx4EXSqx2iooMe9jVvYODgyc3BwmEQRxKDjSpktjxPu6UA83to4WOhKMJpsHBHyJ4jhxgrznTQSEWbG3C0yIDYZMeRfswItuMqIDy/GTi7KDYIvt4grCCQh7hyDJGigcygeKqUJcTVk9AqKXSN7TsMcM8RZi7KE5E6YrY15b+PASx01ed4WMYJ3ZAhYiOytkTUjG4fexocXhaescOR5kM0kiRWwsYNqgJXQCYhY5ZxBXxVKISDOngelKiNGBlkebEypJmaleou4QpSmrLotrdcQbMYGxLJnhUf+nRFTRh+q0CM+uDg9RczJxnVvY+OBWACMqUvHKqQ8lYNIIIL3rMjJooS6JeRqfIJYEmJALQmxfMNjJAy4srpVg1YILkMSZZVtjNy7aBON5+jdRfI9PY57PJaoZTKPIWtM18o1XpuRxzFkrUC2mVkxer5AYVtldStJBiBy+4hdnXkcQjdGELIxWEc8JsSpgP7EHvHhxfqpjb/2ntdrxVcg/gobXxLiS5bqy0t43hbxZbZxuFSUD19ejoiRGbBgw9eM+JJYcZkRf71E7gbEl/CrBBv7bcwzX+Vl89V+ZV/lV4gQI8Qjjw9uGryJeWxfKyLZmIIdSQKxAtmvaomRnBBrIQhx83WLmHUdowjiicdqp27Myc6TjeEmEdkmEA/qJjKPbVtC70KQJWIeATl6d/NicqEAzzZexy3iQIgdIU6yciAtaYUgPc6s0DmvqGrOtzwGYlK3Lnte3CIuR3UrbUAtiyJlh7gjxOBxJyqLqolp1uvxO5soP9m4zlFaIRPKUVrljuEdDLkbWFFlViTIWEbcqtRCGPhYZ4YSORBsnGOeF7beIW47i0mBQ1DKiUJlYAXcE4i1KxmSZqqlUYNUfV5xeDNnL3fLrJBwXITNjBhBqDcdIebGZ8TMbvPjxKMjreB28PyqssjY8G7SaxS3dqduqGmyHjPKK+CeY7YnJMqxWJUsUV4hkLvFru1tfHBT6SfEtYwJhVpGbHjXzzZlK4w1IWsFy6zIiBHYS54RD6xAklEBcZSEGJJhdjwmG4eOojRekyob3yK2hDgWLpWV6BEf3Nzaz48zYlRpTch5BQJ9LLasUGRjNmSbLkdpMC4SYgTDbQThDa2hEOIYbbmzcQOtiG2p7WDjLWLUWTvEJWw8qNs7m2xvIogjrYiyaYOZ5scu58cG+l+15HlqYAWHDmVWaD8gBonbA/lxGvJjU+oqwTCYi+ETTnSBBRrFumeFhR3SgPjQZt8bxGvYuArdYONJfhxaFXirvRo8L0dpJEnQCkc1SNhl9Mgr3AHEOT9GlG4zYr/NKwQtK6QqZB7XAeqGAfSsOLjp+IbHClEaPCbPA2JWrdmkBlEoO4uMGBGw1wrIXjS09mUHGyMKhjGjR7AeESMLNDpnm5BjfFx2o68qLhtCDJhALFvMJ3g+aMWhzc/JlYCY1iss6xo2IN7a2FHV1I5RelA3i5QrOUesaMbcDd6IyjQjFmZqY0Uxr8o8JnVzgx6rxhrQAHNT955nEuqfjPjwJuxe1UQZfY3qUIIVEYhtZbesQOiLbfa8Fn2OiOFk0H2xVTePNzPrK1+lrMcjYtSObY+YqqaGYt7OxqFBqMHUpFw1VUjeRhsf2gx+w4qMOMgmHuIxA2KV1Y0NPI4mI97psep53Nu42rNxzo+RRw51nt1W/4ggjFhh+8rUJdQ6A+JDm9JvYl5HiJNrTI84johlRWlFq8M+KxC4g8vZ5qhuYVpL82ktTeoGMmhKhVjtRhubVnKQsDJsQBxTCCPiv6j+2YC4LGvY2GsgZmZrY+GRIiJZGRD3lWl08Gyq8zTf1kAZ8ViZ1rsI0pgtYlSmld1VhrELqfLZ88AKqh2B+N1N+imPuZXaa8k6DJq1zkCWa6Zq1HTQWAD2bY06j2mfQhFon6MOotO0WROQ8fd5sIVyIQ7Fiuc5gHT3iCsYXKKWTraGYVqUVmrLCtea6HlYI4JKqv5pRePo3cMC0xVvLzvoGa0fQ2poH6Qp9q6GQ9Raj2pBQXerdeqQTya6I0S/IFgwzGsLjeS0flyYDpVo7iS2HY+28PDMDqHQ7+0qFFUTvGLrCK3khWBMtYH20fz7ZxyeWcDP4v/r9d/+35ofvXvMYm/nRkprtBSa9ppo/Tjgd07yn+AC3tFSZ6hYHZIsbE2WKwVCFRhjaBEjJi9Q/RuUyayP77SsyDLDUTEiJysMJtCKqJGKp7HOi2vEvNo4tobrd7xA6cTyCss7xz2miGnNzaI+RpYmkJSnHWKhG+EyYlpuSpr2EinmEeII1W2heiytuUM5bhCAg85KbpzRkQ/+JVKN/BOIVUY81gDCrQMXNXi2RkDoXNECcdRH7x47mcY8KTuEUCsMg+fRzg0hpmQlMJFaHprWkuchE0I1YfNGH+9tzKJSJAFI5XLByjRFkIL2mtKIGKJWUJljTUY8rtFzsw5wXhfZGlkbbIyoGdmA+NDxl+maEGpR2uUiViBM8B4x5blBpKaFjXlGrAixIcSxw5s1xpGcgszB72mzl8JjyojDFjGtZfaIdY84pZ8RI3oDsbBMa3307jGcKStkkqg6Iai6qa1vLFKBIAIveJIIatxVTfK+ojIl9upmKx1Q/ScuLOp3ypGbmCQUApVpRzZuuI5NVUchHO33lMhMUokmXEIuBgUXLDYGwMJaQ799ARWBDx29exxoqsdRdkgdJUN1LBIhHvwWouZbpPuipYUuVsdxTajfv3VaONvAeDT7LBlD6/BU/SP3hOexQY8bjbjTQrBpgUmkUd046h1EICjqWhnW0Z4pEiPZq9u7sP9Lqtt7h6imUbqSHac6r3Iox43Qo41dHVzrneKth+dIZPHIhLrRxmGwsR1sLGDj1tCuQoGETZs+FrYdrSYqpE0UFCEOcrsD2cDyaWJjy9xo4/fPfpGJ/4vZ+N3jZ3v5sV5XBgLrQ0TVbBGM+5RMOBGaunUIzxbOGKJuEY5Beog35MI6ztvUl22ClQJRGvlv4lIj9koX+n3cNcec2VTS7qPmAvUgpkk3XHDjnag16jy4vXbw6VS19dG7x+AmFzRCyjrZyNqGdu40tAK9kwAobw3yIy99WzGbuthHEMtl0h41SNcoY0W//wyKoOqukFG2CohFn212a4HUlYmypqXiJNZidF3nW4+UnnZ5ndWhKLUOsj569zjeFDHTXdlpycFjV5oWmRDMZLqiVVysDcKgkD4gBzIyFW1GDEmDuumuqoWPIjPXdnWCPFUoKBgHYlP1pxWyjVvOaoHAKipvg66RLXKOzM1TuZjWPpIr5FwpHb17LHC67sZLqylFZRWlmSxJJcM2w2KBN6RVtCNA6tbmyjSxREcquFCmTlnrakO7or4yqlPOVMOJkBa2b1A1odJA/zUQj2dY8EHQjdEZFvynY7R3iFh+9O7xxGkEgQNH08lKOCCuJVjRZcSeKkBTBeTqLa27Ja0L3iMGPCZq7pjBGPv1CtAmI6a1L8W1711BojwA4hp1HtkZkX6SyirrYN+M2BdCI3SWR+8ek5wiVqzD62hDs2k7rSsZvBWqKiLkK9VtEEqbqkO6joo30xA+UpMMBseReVAKAmVjbacM79XNm+EMCxeyxuhFo7NRWmfHvZ4cRDBFRG5DeUyVZFP3nnfwuOZe9a9BoURnKOB5NTO0UogsAjUTpVq0ky5zCelhYyX7XYUOdSbyiKAcHEpjMLTQqVJ/viJ6pXvErEVbieo61pF1yFYqt0W8dozT3seaczpsRIVIfyLk8LHRKWLEyo4RYuQ8uiXE7YCYQ+2QpmXEQXmKBRkxUiStrFYIOc5j2k20mtMiS8iIGRCLEbEgxFTjo/4IVWX5DjGPtQlxzS2zDvEKwjmcYTl0fPV9xFRkThGr2HBkScjd6JxQzjYJcc1hY7DCxSZqWuw0XCrTIw4gcejrvBYFNxAjsUuJbMzizsaUsqIWWLN8EAb5WIj26N1jtFPEakTMm5BZEUfEIiNmDeqTirmJjY3WqPUaBOpxlzm5KBPrETdepdgvmbfWEWKmZAod9KSKWxsXeAnqeERpit9FlFLVw5nCQ4d039oYetzbuJ6yAiEKiBV+pV3eLeIUhE7M1go5v6sbaABYYTiU0W1Z0YyscCMrQmfbYFKYsAIpNRDHVgExkgrXe97q/wpxtnH3FvHAY5YRR0IcZF7bBI+DZQp65DQSTmgAoxMhcWfj0GtFKyMhjqRFHRVf2m1NzFUTBWX04DFsHDtWZ1YcPt78BjHFDZYR73keI8RRHUTMLIRCBMfARkcmRGXK+Wjj7SnIXiuAGKWRDa6a2JhOerF8IoQQOyDWOeYdPmb9Vt1QTtPSCq061ZLKYkcrQhSUwGMhDW8ZB3mRCRHiViWIAdQNSb+iBSTknA6Z0GBjVo/qBhvj38IjxkdK6pF7yEkEUY5OjzkWUWPZWlaJaul3jntPLky9RMmDTAiJDVBKiKoMIRWN7ZT1wTWsM6xF/KATel3OhEw+WoA3DoUmSA21SE3Ia0IZcX9OqJKqtYVFaY8aVraQuq2N4ehIJMjzONWriar/PoIcPHY+jXm1rRmqJhMDKOsjwke/ks4xtQhjjlawVRV0tHXR5XW31kkNxJhV7nPZXSaKXKBHPs2LtEz3rEDMU0IXSTXWY8yG+XpcxaLThjziFWtyS9r7b1mfVxw+/j6tQZB4JfKMVjYCzi+l49IoOj3GiNehAWKgR8xD3uj7tU2IFRIZjhFlCKG2LSmKyAfgkFgkFQYety4VPtRgHKmfDWyyPEY9gBUBxXCBpDsMiA8ew58ijqj3LZ0plA2P3nWon6RyNRX1ojYx4vVw8lCz8RSkVqHOx5o442Xtck7D6PyV723ceqPzIihiCWxcg8dIGpxE1TSeVgCHWaBTO3RuE17Qj6PrER/8OsA0P7ZU7xMroBXRgHr4F2Cg6SAWHxFThimLnNRZQlwGzXGV6bJHbHrEdl8reF7xFiSeXzPiOOxAeoNaGoUhBk0LRrG2tHWRjt79WsL+ulsHjkkewFjtKzZ6MyiCcIToL2phWqK66tfdUE+Ax7QPEiPLp3k7g+wW7hBJKzBPaVw/tpLqWdCI9g6QiLTtm4XTgq2NSxE81jYMVdPBr0fsVU2ycwY25kEgmRJbxMkwBLyAqq82pE9I9QuWV1ha8BgjcPB2lrcUrUE4t2WPuM/dcqSo6ZQwig/GSY9rFaPeagXtx7aGhXWEx9IZFjvUeYe/pjG5HJOdhyBHaDHtB3RwOzoSWiDNBNFMEF5y06rWa+gUTXZJtbxD2eeEwHTnei4ho9TO5/Vj3Zl6tPG6hgUkktmoOuQpSZXbs7GOt0DM0AIVIpJ/x0Q4evfrInvnj21nqhqlZddAhJB4G+eFIMSCpxaGlJIRYrNDnMieHeeAKfN3FbroESE4OFV4XiHj789tirSOXBZrQ8fIOwCDI+9O84omgm1r+joAahC8RPQn/w9+beUQ4qpDUmMIceUBv4gNUjL4BhBrQuwOI84xo0vI5ejEh8iIjW4yK3hauwqIRSW1kPDYnxBXqP6BmGwcqh7x4a/PTHcVhOxMZkXV8t7GYEWUmRWamcB9H7dpZy5v66YK8ZsDMS04SJ93FZLnUfO8n1fQ4bKeFcau6Rz9Ouiupd3nQDo/LBVCmjmxQnKmVSw8woBgR+9+jWevlgZimRE3O8SuRwwbE+KcRKgdYuROyIeBmKcBcUk8HhCrLWL6cgwjxGmdWI94W6Un6HHmMUfnsUiEOFemh79KtMeKAXFVGUIsR8RqQGwIcYuMyA6soMMm5HkcRV7Kx2gQz+jkivP5W0IGeryzMbIW9LNOTY94jHmQbyQUhBh5Kqo/Jx10itbdSvpfo1zdPD1uZoevzc3d3cvNg3zeXDfPm3/f3MnN890L+twUm5e7l80Gv95sqteXf08PxeYVDx42zxv8ebjZ3G36129eHp6f7evmuaIP4uMPm/xk8yyf7+6KF7RHT3bz/Dx+oiju0DH+YWsCsCnoI5uHo48fj/4PD5OcEdRIAAA="
seed_bytes = gzip.decompress(base64.b64decode(SEED_GZIP_BASE64))
if hashlib.sha256(seed_bytes).hexdigest() != SEED_SHA256:
    raise ValueError("The embedded alignment checksum does not match.")
seed_text = seed_bytes.decode("utf-8")
seed_alignment = AlignIO.read(io.StringIO(seed_text), "stockholm")
globins = {record.id: str(record.seq).upper().replace(".", "-")
           for record in seed_alignment}
n_cols = seed_alignment.get_alignment_length()
print(f"PF00042.29: {len(globins)} sequences, {n_cols} columns")
print("SHA-256:", SEED_SHA256)
for name in list(globins)[:5]:
    print(name, globins[name][:60])


In [ ]:
import logomaker
import pandas as pd


In [ ]:
import pandas as pd

AAS = "ACDEFGHIKLMNPQRSTVWY"
seqs = list(globins.values())
counts = pd.DataFrame([[sum(s[j] == a for s in seqs) for a in AAS] for j in range(n_cols)],
                      columns=list(AAS), index=range(1, n_cols + 1), dtype=float)
freq = counts.div(counts.sum(axis=1).replace(0, 1), axis=0)                 # residue frequencies, gaps left out
entropy = -(freq * np.log2(freq.where(freq > 0, 1))).sum(axis=1) + 0.0
occupancy = counts.sum(axis=1) / len(seqs)                     # fraction of sequences with a residue
info = (np.log2(20) - entropy) * occupancy                     # information content, lowered by gaps

print(f"entropy ranges from {entropy.min():.2f} to {entropy.max():.2f} bits (maximum possible {np.log2(20):.2f})")
print(f"mean information content per column: {info.mean():.2f} bits")
# Entropy 0 in a column that is mostly gaps just means one or two sequences have a residue there,
# so "fully conserved" also requires every sequence to have a residue in the column.
conserved = entropy.index[np.isclose(entropy, 0) & (occupancy == 1)]
print("fully conserved columns (same residue in every sequence):", list(conserved))

In [ ]:
import logomaker

heights = freq.mul(info, axis=0)
rows = [(1, 47), (48, 94), (95, n_cols)]
fig, axes = plt.subplots(len(rows), 1, figsize=(15, 7.5))
for ax, (a, b) in zip(axes, rows):
    logomaker.Logo(heights.loc[a:b], ax=ax, color_scheme="chemistry", vpad=0.05, width=0.9)
    ax.set_xlim(a - 0.6, b + 0.6)
    ax.set_ylim(0, 4.6)
    ax.set_ylabel("bits")
axes[-1].set_xlabel("alignment column")
fig.suptitle(f"Pfam Globin seed alignment, {len(seqs)} sequences")
plt.tight_layout()
plt.show()

### Mapping columns to human HBB

Alignment-column numbers describe this saved alignment. To relate them to HBB, align the alligator HBB reference row to the human sequence and transfer its coordinates.

The table reports both mature HBB numbering and full UniProt sequence numbering. In this HBB sequence, the initiator methionine makes the UniProt position one greater than the mature-chain position. This is a mapping for this reference and sequence.


In [ ]:
HBB_HUMAN = ("MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTF"
             "ATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH")   # UniProt P68871

ref_name = next(n for n in globins if n.startswith("HBB_ALLMI"))
ref_row = globins[ref_name]
ref_seq = ref_row.replace("-", "")
S_map, tr_map = global_align(ref_seq, HBB_HUMAN, -2.0, lambda a, b: 1.0 if a == b else -1.0)
ali_ref, ali_hbb = traceback(ref_seq, HBB_HUMAN, S_map, tr_map)

# residue index in ref_seq -> residue index in HBB_HUMAN (index 0 is the Met, so an index is the mature number)
ref_to_hbb, i, j = {}, 0, 0
for a, b in zip(ali_ref, ali_hbb):
    if a != "-" and b != "-":
        ref_to_hbb[i] = j
    i += a != "-"
    j += b != "-"
col_to_hbb, k = {}, -1
for c, ch in enumerate(ref_row, start=1):
    if ch != "-":
        k += 1
        if k in ref_to_hbb:
            col_to_hbb[c] = ref_to_hbb[k]

print("The 10 columns with the highest information content:")
print(f"{'column':>6} {'bits':>5}  most common residue    human HBB")
for c in info.sort_values(ascending=False).index[:10]:
    top = freq.loc[c].idxmax()
    n_top = int(counts.loc[c, top])
    h = col_to_hbb.get(c)
    label = f"{HBB_HUMAN[h]}{h} mature / {HBB_HUMAN[h]}{h + 1} UniProt" if h is not None else "-"
    print(f"{c:6d} {info[c]:5.2f}  {top} in {n_top:2d} of {int(counts.loc[c].sum()):2d}        {label}")

Use the table to identify the two positions conserved across this saved alignment. Check their roles in the chapter and the [HBB record](https://www.uniprot.org/uniprotkb/P68871/entry). In HBB, proximal His92 coordinates the haem iron. Conservation in the sampled alignment and a known structural role provide complementary evidence.

### Optional small-sample comparison

The next cell draws five sequences with a fixed random seed. Compare their apparent conservation with the full snapshot. Change the seed to examine a different subset and keep a record of your selection.


In [ ]:
# Challenge 1: the same logo from only 5 random sequences
rng = np.random.default_rng(2027)
few = [seqs[i] for i in rng.choice(len(seqs), size=5, replace=False)]
counts5 = pd.DataFrame([[sum(s[j] == a for s in few) for a in AAS] for j in range(n_cols)],
                       columns=list(AAS), index=range(1, n_cols + 1), dtype=float)
freq5 = counts5.div(counts5.sum(axis=1).replace(0, 1), axis=0)
entropy5 = -(freq5 * np.log2(freq5.where(freq5 > 0, 1))).sum(axis=1)
info5 = (np.log2(20) - entropy5) * counts5.sum(axis=1) / len(few)
print("fully conserved columns in this sample of 5:", int(((entropy5 == 0) & (counts5.sum(axis=1) == 5)).sum()),
      f" (in all {len(seqs)}: {len(conserved)})")
fig, ax = plt.subplots(figsize=(15, 2.5))
logomaker.Logo(freq5.mul(info5, axis=0).loc[1:47], ax=ax, color_scheme="chemistry", width=0.9)
ax.set_ylim(0, 4.6); ax.set_ylabel("bits"); ax.set_title("Columns 1–47, fixed sample of five globins")
plt.show()

## Review and submit

Check your row entries, traceback, identity and similarity counts, and entropy calculations. Record the scoring settings and the PF00042.29 snapshot version with your results. Review the ten quiz items in Canvas and complete the feedback survey according to the course arrangements.
